# MP1 · Prompt Lab — Compare LLM Strategies on a Task

**Starter template.** Fill in the TODOs. ~5-8 hours over 3 days.

Read `learner/MP1_Brief.md` before starting if you haven't already.

---

## Setup

In [1]:
import asyncio
import json
import re
import os
import pickle
import time
from itertools import product
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd
from openai import AsyncOpenAI
from pydantic import BaseModel, Field, ValidationError

load_dotenv()

# Make sure your OPENAI_API_KEY is set in the environment
assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY first'

client = AsyncOpenAI()

MODEL = 'gpt-4o-mini'
JUDGE_MODEL = 'gpt-4o'
TEMPERATURE = 0.0

# Cost rates ($ per token) — from W4 cost.py
RATES = {
    'gpt-4o-mini': {'in': 0.15 / 1_000_000, 'out': 0.60 / 1_000_000},
    'gpt-4o':      {'in': 2.50 / 1_000_000, 'out': 10.00 / 1_000_000},
}

print('Setup complete.')

Setup complete.


## Step 1 — Load the data

In [2]:
DATA_DIR = Path('data')   # adjust if your folder layout differs

snippets = [json.loads(line) for line in (DATA_DIR / 'job_snippets.jsonl').read_text().splitlines() if line.strip()]
golden = {row['id']: row for row in (json.loads(line) for line in (DATA_DIR / 'golden_set.jsonl').read_text().splitlines() if line.strip())}

print(f'Loaded {len(snippets)} snippets, {len(golden)} golden entries.')
print('Sample snippet:', snippets[0])

Loaded 10 snippets, 10 golden entries.
Sample snippet: {'id': 'j01', 'snippet': 'Acme Corp is hiring a Senior Software Engineer to join our platform team. The ideal candidate has 5+ years of backend development experience and strong skills in Python and distributed systems.'}


## Step 2 — Write the four prompt strategies

Each strategy is a function that takes a snippet text and returns the messages list to send to the LLM.

Implement all four. Keep each one focused — the point is to *see* the difference between strategies, not to over-engineer any one.

**TODO:** fill in the four `prompt_*` functions below.

In [3]:
class JobListing(BaseModel):
    company: str = Field(description="Company Name")
    role: str = Field(description="The Job Title")
    # LLM predicts word by word. So reasoning notes comes first and then the actual derived value
    # Though notes appear first in the order, without explicit instruction, it didn't reason first and years of experience came in front.
    notes: str | None = Field(description="Optional Steps for deriving years of experience", default = None)
    years_experience_required: int | None = Field(description="The minimum years of experience required")

format_instructions = json.dumps(JobListing.model_json_schema(), indent=2)
format_instructions

'{\n  "properties": {\n    "company": {\n      "description": "Company Name",\n      "title": "Company",\n      "type": "string"\n    },\n    "role": {\n      "description": "The Job Title",\n      "title": "Role",\n      "type": "string"\n    },\n    "notes": {\n      "anyOf": [\n        {\n          "type": "string"\n        },\n        {\n          "type": "null"\n        }\n      ],\n      "default": null,\n      "description": "Optional Steps for deriving years of experience",\n      "title": "Notes"\n    },\n    "years_experience_required": {\n      "anyOf": [\n        {\n          "type": "integer"\n        },\n        {\n          "type": "null"\n        }\n      ],\n      "description": "The minimum years of experience required",\n      "title": "Years Experience Required"\n    }\n  },\n  "required": [\n    "company",\n    "role",\n    "years_experience_required"\n  ],\n  "title": "JobListing",\n  "type": "object"\n}'

In [4]:
def prompt_zero_shot(snippet_text: str) -> list[dict]:
    """Strategy 1 — zero-shot. Just ask, no examples, no persona."""
    return [
        {'role': 'system', 'content': 'Extract the following details from the job posting in json format with fields `company` (which is doing the hiring), `role` (job title) & `years_experience_required` (years of minimum experience required only if available)'},
        {'role': 'user', 'content': snippet_text }
    ]

def prompt_few_shot(snippet_text: str) -> list[dict]:
    """Strategy 2 — few-shot. Include 2-3 worked examples in the prompt."""
    return [
        {'role': 'system', 'content': 'Extract the following details from the job posting in json format with fields `company` (which is doing the hiring), `role` (job title) & `years_experience_required` (years of minimum experience required only if available)'},
        {'role': 'user', 'content': 'XYZ Inc. is hiring for a Senior Buisness Analyst. Around five years of experience is required in the Financial Crime & Compliance Stream' },
        {'role': 'assistant', 'content': '{"company": "XYZ Inc.", "role" : "Senior Business Analyst", "years_experience_required" : 5 }' },
        {'role': 'user', 'content': 'Chameleon is looking for a Vice President with demonstrated experience in leading Payments processing Stream. Ideal candidates should have 12+ years of experience with a minimum of 6+ years in product management role' },
        {'role': 'assistant', 'content': '{"company": "Chameleon", "role" : "Vice President", "years_experience_required" : 12 }' },
        {'role': 'user', 'content': 'Postion of Architect is open in CamelCase. Candidate profiles with demonstrated ability will be considered' },
        {'role': 'assistant', 'content': '{"company": "CamelCase", "role" : "Architect", "years_experience_required" : null }' },
        {'role': 'user', 'content': snippet_text }
    ]

def prompt_structured(snippet_text: str) -> list[dict]:
    """Strategy 3 — structured / role-based. Use a system prompt with a persona and explicit JSON schema."""
    return [
        {'role': 'system', 'content': f"""
            You are an expert job posting analyst who inventories the required metrics from all postings. 
            Extract the following details from the job posting: company (which is doing the hiring), job title & an years of minimum experience required only if available.
            Hard Rules:
            Do not invent any values which is not available in the context.
            Output Format:
            Return a JSON object that conforms to the below schema, with only the keys company, role and years_experience_required. Do not repeat the schema.
            ```
            {format_instructions}
            ```
            """
        },
        {'role': 'user', 'content': snippet_text }
    ]


def prompt_cot(snippet_text: str) -> list[dict]:
    """Strategy 4 — chain-of-thought. Ask the model to reason before answering."""
    return [
        {'role': 'system', 'content': f"""
            You are an expert job posting analyst who looks for required metrics of all postings. 
            Extract the following details from the job posting in json format: company (which is doing the hiring), job title & years of minimum experience required only if available. 
            Derive the minimum experience required step by step and capture the steps in the notes field of the output json. Do the reasoning before producing the output for years of minimum experience required
            Hard Rules:
            Do not invent any values which is not available in the context.
            Output Format:
            Return a JSON object that conforms to the below schema, with only the keys company, role and years_experience_required. Do not repeat the schema. 
            ```
            {format_instructions}
            ```
            """
        },
        {'role': 'user', 'content': snippet_text }
    ]

STRATEGIES = {
    'zero_shot': prompt_zero_shot,
    'few_shot': prompt_few_shot,
    'structured': prompt_structured,
    'cot': prompt_cot,
}

## Step 3 — Async batching

Run all 10 snippets × 4 strategies = 40 calls in parallel.

Capture for each call: strategy, snippet_id, raw response, parsed extraction, cost, latency.

**TODO:** implement `run_one` (single call) and `run_all` (batch all 40).

In [5]:
def parse_response(text: str) -> dict | None:
    """Try to parse a JSON object out of the model's response. Return None if it doesn't parse.
    
    Hint: models sometimes wrap JSON in ```json ... ``` fences. Strip them first.
    """

    # 1. Extracting content inside ```json ... ``` or ``` ... ``` blocks
    match = re.search(r"```(?:json)?\s*([\s\S]*?)\s*```", text)
    if match:
        clean_text = match.group(1)
    else:
        # 2. Fallback: find the first '{' and the last '}'
        match = re.search(r"(\{[\s\S]*\})", text)
        clean_text = match.group(1) if match else None
    
    try:
        parsed_response = json.loads(clean_text.strip()) if clean_text else None
        return parsed_response if type(parsed_response) is dict else None
    except json.JSONDecodeError:
            return None


async def run_one(strategy_name: str, snippet: dict) -> dict:
    """Run one strategy on one snippet. Return a dict with all the captured fields."""
    start_time = time.time()
    
    # Standard pricing structure (e.g., per token)
    model_pricing = RATES[MODEL]
    
    # Make the API call using your client & model
    response = await client.chat.completions.create(
        model=MODEL,
        messages=STRATEGIES[strategy_name](snippet["snippet"]),
        temperature=TEMPERATURE
    )
    
    elapsed_time = time.time() - start_time
    response_text = response.choices[0].message.content if len(response.choices) > 0 else ''
    usage = response.usage
    total_prompt_tokens = usage.prompt_tokens
    completion_tokens = usage.completion_tokens

    cost = total_prompt_tokens * model_pricing.get("in", 0) 
    cost += completion_tokens * model_pricing.get("out", 0)
    result = {
        'strategy': strategy_name,
        'snippet_id': snippet["id"],
        'raw_response': response_text,
        'extracted': parse_response(response_text),
        'cost_usd': cost,
        'latency_s': elapsed_time
    }
    return result


async def run_all() -> list[dict]:
    """Run all 10 × 4 = 40 calls in parallel. Use asyncio.gather."""
    # product() is the cartesian product, so no nested for-loops:
    # 4 strategies × 10 snippets = 40 (strategy_name, snippet) pairs.
    # Iterating a dict yields its keys, so `STRATEGIES` gives the strategy names.
    # Calling an async function does NOT run it; it only creates a coroutine object.
    coros = [run_one(name, snippet) for name, snippet in product(STRATEGIES, snippets)]

    # gather() wraps each coroutine in a Task and schedules all 40 on the event loop.
    # Each task runs until its `await client.chat.completions.create(...)`, then yields,
    # so the 40 network waits overlap: total time ≈ the slowest single call, not the sum.
    # Results come back in the order of `coros`, not the order the calls finished.
    # Transient 429/5xx errors are retried by the OpenAI client itself (max_retries=2).
    return await asyncio.gather(*coros)

In [6]:
result = await run_one("zero_shot", snippets[0])
print(json.dumps(result, indent=1))

{
 "strategy": "zero_shot",
 "snippet_id": "j01",
 "raw_response": "```json\n{\n  \"company\": \"Acme Corp\",\n  \"role\": \"Senior Software Engineer\",\n  \"years_experience_required\": 5\n}\n```",
 "extracted": {
  "company": "Acme Corp",
  "role": "Senior Software Engineer",
  "years_experience_required": 5
 },
 "cost_usd": 3.45e-05,
 "latency_s": 2.4850406646728516
}


In [7]:
result = await run_one("structured", snippets[9])
print(json.dumps(result, indent=1))

{
 "strategy": "structured",
 "snippet_id": "j10",
 "raw_response": "{\n  \"company\": \"Cyberdyne Systems\",\n  \"role\": \"AI/ML Research Scientist\",\n  \"years_experience_required\": null\n}",
 "extracted": {
  "company": "Cyberdyne Systems",
  "role": "AI/ML Research Scientist",
  "years_experience_required": null
 },
 "cost_usd": 7.845e-05,
 "latency_s": 1.2147815227508545
}


In [8]:
result = await run_one("cot", snippets[9])
print(json.dumps(result, indent=1))

{
 "strategy": "cot",
 "snippet_id": "j10",
 "raw_response": "{\n  \"company\": \"Cyberdyne Systems\",\n  \"role\": \"AI/ML Research Scientist\",\n  \"years_experience_required\": null,\n  \"notes\": \"The job posting does not specify a minimum years of experience requirement. It states that they hire based on demonstrated impact rather than a specific number of years.\"\n}",
 "extracted": {
  "company": "Cyberdyne Systems",
  "role": "AI/ML Research Scientist",
  "years_experience_required": null,
  "notes": "The job posting does not specify a minimum years of experience requirement. It states that they hire based on demonstrated impact rather than a specific number of years."
 },
 "cost_usd": 0.00010559999999999999,
 "latency_s": 2.032106637954712
}


In [9]:
# Run once and cache. Re-running the notebook, or a kernel restart, should not pay for
# 40 identical calls, and a cache keeps the scores reproducible (temperature 0 is not
# a guarantee of identical outputs). Delete results.pkl to force a fresh run.
RESULTS_CACHE = Path('results.pkl')

if RESULTS_CACHE.exists():
    results = pickle.loads(RESULTS_CACHE.read_bytes())
    print(f'Loaded {len(results)} cached results from {RESULTS_CACHE}.')
else:
    wall_start = time.perf_counter()
    results = await run_all()
    wall_s = time.perf_counter() - wall_start
    RESULTS_CACHE.write_bytes(pickle.dumps(results))
    # The gap between these two numbers is what asyncio.gather bought us.
    print(f'Ran {len(results)} calls in {wall_s:.1f} s wall time '
          f'(sum of per-call latencies: {sum(r["latency_s"] for r in results):.1f} s).')

results[0]

Ran 40 calls in 3.6 s wall time (sum of per-call latencies: 107.2 s).


{'strategy': 'zero_shot',
 'snippet_id': 'j01',
 'raw_response': '```json\n{\n  "company": "Acme Corp",\n  "role": "Senior Software Engineer",\n  "years_experience_required": 5\n}\n```',
 'extracted': {'company': 'Acme Corp',
  'role': 'Senior Software Engineer',
  'years_experience_required': 5},
 'cost_usd': 3.45e-05,
 'latency_s': 2.047515630722046}

In [10]:
results[1]

{'strategy': 'zero_shot',
 'snippet_id': 'j02',
 'raw_response': '```json\n{\n  "company": "Northwind Ltd.",\n  "role": "Data Analyst",\n  "years_experience_required": 2\n}\n```',
 'extracted': {'company': 'Northwind Ltd.',
  'role': 'Data Analyst',
  'years_experience_required': 2},
 'cost_usd': 3.39e-05,
 'latency_s': 3.0412988662719727}

## Step 4 — Score against the golden set

Three scores per (strategy × snippet) pair:

1. **accuracy** — how many of 3 fields match (0, 1, 2, or 3)?
2. **parse_success** — did the response parse cleanly?
3. **llm_judge_score** — 1-4 score from gpt-4o-as-judge

**TODO:** implement the three score functions.

In [11]:
FIELDS = ('company', 'role', 'years_experience_required')

# A sentinel that can never equal a real value. A key the model left out must count as
# wrong, even when the gold value is null; dict.get(field) would return None and
# silently "match" a null gold value.
_MISSING = object()


def _normalise(value: object) -> object:
    """Casefold strings and collapse whitespace; leave numbers and None as they are."""
    if isinstance(value, str):
        return ' '.join(value.split()).casefold()
    return value


def score_accuracy(extracted: dict | None, gold: dict) -> int:
    """Compare 3 fields. Case-insensitive, whitespace-trimmed for strings. Return 0, 1, 2, or 3."""
    if extracted is None:
        return 0
    # Each comparison is a bool, and bool is a subclass of int, so sum() counts the Trues.
    # Deliberately strict: "5" (a string) does not match 5, and "3-5" does not match 3.
    return sum(
        _normalise(extracted.get(field, _MISSING)) == _normalise(gold[field])
        for field in FIELDS
    )


JUDGE_SYSTEM_PROMPT = """
You grade the output of an information-extraction system that reads job postings and
extracts three fields: company, role, and years_experience_required (the minimum years
of experience, or null when the posting states none).

You get the job snippet, a reference answer, annotator notes on what is acceptable, and
the extracted answer. Judge each field against the snippet and the reference. Treat
differences of case, punctuation or spacing as correct. A missing field is wrong.
A value the snippet does not support is fabricated; in particular, a number of years
when the posting states no requirement is a fabrication.

Rubric:
4 - all three fields correct
3 - two of three correct, no fabricated data
2 - one of three correct, or any field fabricated
1 - none correct

Reply with JSON only, reasoning first so that the score follows from it:
{"reasoning": "<one or two sentences>", "score": <1, 2, 3 or 4>}
"""


async def score_llm_judge(snippet_text: str, extracted: dict | None, gold: dict) -> int:
    """Use gpt-4o as a judge. Return integer 1-4.

    Rubric (suggested):
      4 — all three fields correct
      3 — two of three correct, no fabricated data
      2 — one of three correct, or fabricated a field
      1 — none correct or unparsable
    """
    if extracted is None:
        return 1  # the rubric fixes unparsable at 1, so there is nothing to pay a judge for

    reference = {field: gold[field] for field in FIELDS}
    user_prompt = (
        f'Job snippet:\n{snippet_text}\n\n'
        f'Reference answer:\n{json.dumps(reference)}\n'
        f'Annotator notes: {gold["notes"]}\n\n'
        f'Extracted answer:\n{json.dumps(extracted)}'
    )
    # JSON mode is fine here: the judge is measuring tool, not one of the strategies under
    # test, so constraining its output format does not bias the prompt comparison.
    response = await client.chat.completions.create(
        model=JUDGE_MODEL,
        temperature=0.0,
        response_format={'type': 'json_object'},
        messages=[
            {'role': 'system', 'content': JUDGE_SYSTEM_PROMPT},
            {'role': 'user', 'content': user_prompt},
        ],
    )
    verdict = parse_response(response.choices[0].message.content or '')
    score = verdict.get('score') if verdict else None
    # Fail loudly rather than guess: a silently defaulted score would corrupt the table.
    if score not in {1, 2, 3, 4}:
        raise ValueError(f'Judge returned an unusable verdict: {response.choices[0].message.content!r}')
    return score

In [12]:
def is_schema_valid(extracted: dict | None) -> bool:
    """True if the extraction is a usable JobListing, not just valid JSON.

    parse_success only asks "is it JSON?". An echoed schema is valid JSON but useless,
    so this second check asks "does it have the right fields with the right types?".
    strict=True turns off Pydantic's type coercion, so "5" (a string) fails, which
    matches score_accuracy. Extra keys such as `notes` are ignored; an echoed schema
    fails because it has no top-level company or role.
    """
    if extracted is None:
        return False
    try:
        JobListing.model_validate(extracted, strict=True)
    except ValidationError:
        return False
    return True


async def score_all(results: list[dict]) -> list[dict]:
    """Attach accuracy, parse_success and llm_judge_score to every result row."""
    snippet_text = {s['id']: s['snippet'] for s in snippets}

    # The judge calls are I/O-bound too, so run them concurrently, same as run_all.
    judge_scores = await asyncio.gather(*(
        score_llm_judge(snippet_text[r['snippet_id']], r['extracted'], golden[r['snippet_id']])
        for r in results
    ))

    # {**r, ...} copies the row and adds the new keys, leaving `results` itself unchanged.
    # strict=True makes zip raise if the two lists ever differ in length.
    return [
        {
            **r,
            'accuracy': score_accuracy(r['extracted'], golden[r['snippet_id']]),
            'parse_success': r['extracted'] is not None,
            'schema_valid': is_schema_valid(r['extracted']),
            'llm_judge_score': judge,
        }
        for r, judge in zip(results, judge_scores, strict=True)
    ]


# Same caching idea as results.pkl: the judge costs money and is not perfectly repeatable.
SCORED_CACHE = Path('scored.pkl')

if SCORED_CACHE.exists():
    scored = pickle.loads(SCORED_CACHE.read_bytes())
    print(f'Loaded {len(scored)} scores from scored.pkl.')
else:
    scored = await score_all(results)
    SCORED_CACHE.write_bytes(pickle.dumps(scored))
    print(f'Scored {len(scored)} results.')

Scored 40 results.


## Step 5 — Build the comparison table

In [13]:
df = pd.DataFrame(scored)

summary = df.groupby('strategy').agg({
    'accuracy': 'mean',
    'parse_success': 'mean',
    'schema_valid': 'mean',
    'llm_judge_score': 'mean',
    'cost_usd': 'sum',
    'latency_s': 'median',
}).reindex(list(STRATEGIES))  # keep the strategies in the order they were defined
# One round(3) for everything would show the cost as 0.0: 40 gpt-4o-mini calls cost
# fractions of a cent. Round cost to 6 places and everything else to 3.
summary = summary.round({'accuracy': 3, 'parse_success': 3, 'schema_valid': 3, 'llm_judge_score': 3,
                         'cost_usd': 6, 'latency_s': 3})

summary.columns = ['Accuracy (mean of 3)', 'Parse rate', 'Schema-valid rate', 'Judge score', 'Total cost ($)', 'Latency p50 (s)']
summary

,Accuracy (mean of 3),Parse rate,Schema-valid rate,Judge score,Total cost ($),Latency p50 (s)
strategy,,,,,,
zero_shot,2.7,1.0,1.0,3.9,0.000353,2.712
few_shot,2.8,1.0,1.0,4.0,0.000585,2.520
structured,2.8,1.0,1.0,3.9,0.000766,2.511
cot,2.7,1.0,1.0,3.9,0.001028,2.919


### Where the strategies differ

Accuracy (0–3) for every snippet × strategy. The means above hide which snippets are hard;
this shows them.

In [14]:
df.pivot(index='snippet_id', columns='strategy', values='accuracy')[list(STRATEGIES)]

strategy,zero_shot,few_shot,structured,cot
snippet_id,,,,
j01,3,3,3,3
j02,2,2,3,2
j03,3,3,3,3
j04,3,3,3,3
j05,2,3,2,2
j06,3,3,3,3
j07,3,3,3,3
j08,2,2,2,2
j09,3,3,3,3


### What each strategy extracted for years of experience

The years field is where the snippets are deliberately tricky (fuzzy wording, a range,
spelled-out numbers, "fresh grads", and no requirement at all in j10).

In [15]:
# json.dumps shows null for None and keeps the quotes on a string such as "5", so a
# string answer (which score_accuracy counts as wrong) is visible at a glance.
years = df.assign(
    years=df['extracted'].map(
        lambda e: json.dumps(e.get('years_experience_required', 'MISSING')) if e else 'UNPARSED'
    )
).pivot(index='snippet_id', columns='strategy', values='years')[list(STRATEGIES)]
years.insert(0, 'gold', [json.dumps(golden[sid]['years_experience_required']) for sid in years.index])
years

strategy,gold,zero_shot,few_shot,structured,cot
snippet_id,,,,,
j01,5,5,5,5,5
j02,2,2,2,2,2
j03,4,4,4,4,4
j04,6,6,6,6,6
j05,0,null,0,null,null
j06,7,7,7,7,7
j07,3,3,3,3,3
j08,5,5,5,5,5
j09,3,3,3,3,3


### Where the deterministic score and the judge disagree

Accuracy 3 should go with judge 4. Rows listed here are worth reading: either the exact-match
rule is too strict (for example on role wording) or the judge is too lenient.

In [16]:
expected_judge = df['accuracy'].map({3: 4, 2: 3, 1: 2, 0: 1})
disagree = df[df['llm_judge_score'] != expected_judge]
disagree[['strategy', 'snippet_id', 'extracted', 'accuracy', 'llm_judge_score']]

,strategy,snippet_id,extracted,accuracy,llm_judge_score
1,zero_shot,j02,"{'company': 'Northwind Ltd.', 'role': 'Data An...",2,4
7,zero_shot,j08,"{'company': 'Wonka Confectionery Ltd.', 'role'...",2,4
11,few_shot,j02,"{'company': 'Northwind Ltd.', 'role': 'Data An...",2,4
17,few_shot,j08,"{'company': 'Wonka Confectionery Ltd.', 'role'...",2,4
27,structured,j08,"{'company': 'Wonka Confectionery Ltd.', 'role'...",2,4
31,cot,j02,"{'company': 'Northwind Ltd.', 'role': 'Data An...",2,4
37,cot,j08,"{'company': 'Wonka Confectionery Ltd.', 'role'...",2,4


In [17]:
golden['j08']

{'id': 'j08',
 'company': 'Wonka Confectionery Ltd',
 'role': 'Senior UX Researcher',
 'years_experience_required': 5,
 'notes': "clean — 'five years' → 5"}

In [18]:
results[27]["extracted"]

{'company': 'Wonka Confectionery Ltd.',
 'role': 'Senior UX Researcher',
 'years_experience_required': 5}

## Step 6 — Write your reflection

Open `mp1_writeup.md` and answer the four questions from the brief.

Then commit:

```bash
git add mp1/
git commit -m 'feat(mp1): prompt strategy comparison + writeup'
```